# Optimal Traveler Strategy Tool & Interactive Dashboard

## Setup & Data Loading

In [37]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# ── Load & prepare data ──────────────────────────────────────────────
df = pd.read_excel('airline_ticket_dataset.xlsx')

lcc_airlines = ['WN', 'NK', 'F9', 'G4', 'SY', 'B6']

df['lcc_present'] = (
    df['carrier_lg'].isin(lcc_airlines) | df['carrier_low'].isin(lcc_airlines)
)
df['market_structure'] = pd.cut(
    df['large_ms'],
    bins=[0, 0.5, 0.75, 1.0],
    labels=['Competitive (<50%)', 'Moderate (50-75%)', 'Dominant (>75%)']
)
df['fare_per_mile'] = df['fare'] / df['nsmiles']
df['price_per_mile'] = df['fare_per_mile']  # Alias for strategy tool
df['dominant_type'] = np.where(
    df['carrier_lg'].isin(lcc_airlines), 'LCC', 'Legacy'
)
df['route'] = df['city1'] + '  ↔  ' + df['city2']
df['route_directional'] = df['city1'] + ' → ' + df['city2']
df['lcc_count'] = (
    df['carrier_lg'].isin(lcc_airlines).astype(int) +
    df['carrier_low'].isin(lcc_airlines).astype(int)
)
df['city_pair'] = df.apply(lambda row: ' ↔ '.join(sorted([row['city1'], row['city2']])), axis=1)

# ── Train predictive model (for Fare Predictor panel) ────────────────
features = ['nsmiles', 'large_ms', 'lf_ms', 'passengers', 'quarter']
ml_data = df.dropna(subset=features + ['fare'])
X = ml_data[features]
y = ml_data['fare']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# ── Precompute dropdown options ──────────────────────────────────────
all_cities = sorted(set(df['city1'].unique()) | set(df['city2'].unique()))
all_carriers = sorted(
    set(df['carrier_lg'].dropna().unique()) | set(df['carrier_low'].dropna().unique())
)

print(f"Loaded {len(df):,} routes across {len(all_cities)} cities")
print(f"Random Forest model trained (R-squared = {rf_model.score(X_test, y_test):.3f})")
print("Data loaded successfully!")

Loaded 14,004 routes across 136 cities
Random Forest model trained (R-squared = 0.787)
Data loaded successfully!


## Available Cities in Dataset

Let's see what cities are available so you know what to input!

In [38]:
print(f"Dataset contains {len(all_cities)} cities\n")
print("=" * 80)
print("AVAILABLE CITIES (copy/paste the exact name when inputting):")
print("=" * 80)

# Display in columns for easier reading
for i, city in enumerate(all_cities, 1):
    print(f"{i:3d}. {city}")
    
print("\n" + "=" * 80)
print("Note: Be sure to copy the exact city name (including state/area designation)")
print("=" * 80)

Dataset contains 136 cities

AVAILABLE CITIES (copy/paste the exact name when inputting):
  1. Albany, NY
  2. Albuquerque, NM
  3. Allentown/Bethlehem/Easton, PA
  4. Amarillo, TX
  5. Appleton, WI
  6. Asheville, NC
  7. Aspen, CO
  8. Atlanta, GA (Metropolitan Area)
  9. Atlantic City, NJ
 10. Austin, TX
 11. Bangor, ME
 12. Belleville, IL
 13. Bellingham, WA
 14. Bend/Redmond, OR
 15. Billings, MT
 16. Birmingham, AL
 17. Bismarck/Mandan, ND
 18. Boise, ID
 19. Boston, MA (Metropolitan Area)
 20. Bozeman, MT
 21. Buffalo, NY
 22. Burlington, VT
 23. Cedar Rapids/Iowa City, IA
 24. Charleston, SC
 25. Charlotte, NC
 26. Charlottesville, VA
 27. Chicago, IL
 28. Cincinnati, OH
 29. Cleveland, OH (Metropolitan Area)
 30. Colorado Springs, CO
 31. Columbia, SC
 32. Columbus, OH
 33. Dallas/Fort Worth, TX
 34. Dayton, OH
 35. Denver, CO
 36. Des Moines, IA
 37. Detroit, MI
 38. Eagle, CO
 39. El Paso, TX
 40. Eugene, OR
 41. Eureka/Arcata, CA
 42. Everett, WA
 43. Fargo, ND
 44. Fayette

## Affordability Score for All Flights
1. Competition Score (0-25 points)
- Measures how many low-cost carriers fly the route
- More LCCs = higher score = better for you
- Calculation: Routes with the highest LCC market share get 25 points, others are scaled proportionally

2. Efficiency Score (0-25 points)
- Measures cost per mile traveled
- Lower $/mile = higher score = better value
- Calculation: Compares route's price-per-mile to the 95th percentile (filters out extreme outliers), then inverts it so cheaper routes score higher

3. Market Power Score (0-25 points)
- Measures whether big airlines dominate the route
- Less dominance = higher score = better prices
- Calculation: Takes 1 minus the large carrier market share. If one airline has 90% control, you get low points. If it's spread among many carriers, you get high points.
- Price Score (0-25 points)

4. Measures the actual ticket price
- Lower fare = higher score
- Calculation: Compares route's fare to the 95th percentile, then inverts it so cheaper fares score higher

In [39]:
def calculate_route_scores(df_input):
    """
    Calculate affordability scores for each route.
    Higher score = better deal for travelers.
    """
    df_scored = df_input.copy()
    
    # 1. Competition Score (0-25 points): Based on LCC market share
    df_scored['competition_score'] = (df_scored['lf_ms'] / df_scored['lf_ms'].max()) * 25
    
    # 2. Efficiency Score (0-25 points): Based on price per mile (inverted)
    max_ppm = df_scored['price_per_mile'].quantile(0.95)  # Use 95th percentile to avoid outliers
    df_scored['efficiency_score'] = (1 - (df_scored['price_per_mile'] / max_ppm)) * 25
    df_scored['efficiency_score'] = df_scored['efficiency_score'].clip(0, 25)
    
    # 3. Market Power Score (0-25 points): Lower carrier dominance = better
    df_scored['market_score'] = (1 - df_scored['large_ms']) * 25
    
    # 4. Price Score (0-25 points): Lower absolute fare = better
    max_fare = df_scored['fare'].quantile(0.95)
    df_scored['price_score'] = (1 - (df_scored['fare'] / max_fare)) * 25
    df_scored['price_score'] = df_scored['price_score'].clip(0, 25)
    
    # Total Affordability Score (0-100)
    df_scored['affordability_score'] = (
        df_scored['competition_score'] + 
        df_scored['efficiency_score'] + 
        df_scored['market_score'] + 
        df_scored['price_score']
    )
    
    return df_scored

# Apply scoring
df_scored = calculate_route_scores(df)

print("Route scoring complete!")
print(f"\n Score Distribution:")
print(f"   Mean: {df_scored['affordability_score'].mean():.1f}")
print(f"   Median: {df_scored['affordability_score'].median():.1f}")
print(f"   Best possible route: {df_scored['affordability_score'].max():.1f}")
print(f"   Worst route: {df_scored['affordability_score'].min():.1f}")

Route scoring complete!

 Score Distribution:
   Mean: 41.2
   Median: 42.7
   Best possible route: 65.9
   Worst route: 2.1


### Top 30 Best Value Routes by Affordability Score

Here are the overall best routes in the dataset - use these for inspiration!

In [40]:
# Get best routes
best_routes = df_scored.groupby('route_directional').agg({
    'affordability_score': 'mean',
    'fare': 'mean',
    'price_per_mile': 'mean',
    'lf_ms': 'mean',
    'nsmiles': 'mean',
    'passengers': 'sum'
}).reset_index().nlargest(30, 'affordability_score')

fig = px.bar(best_routes, 
             x='affordability_score', 
             y='route_directional',
             orientation='h',
             title='Top 30 Best Value Routes (Affordability Score)',
             labels={'affordability_score': 'Affordability Score (0-100)', 'route_directional': 'Route'},
             color='fare',
             color_continuous_scale='RdYlGn_r',
             hover_data={'fare': ':.2f', 'lf_ms': ':.1%', 'price_per_mile': ':.3f', 'nsmiles': ':.0f'})

fig.update_layout(height=900, yaxis={'categoryorder':'total ascending'})
fig.show()

print("\n These routes offer the best combination of:")
print("   ✓ Strong competition (high LCC presence)")
print("   ✓ Low market concentration")
print("   ✓ Efficient pricing (good $/mile)")
print("   ✓ Affordable absolute fares")


 These routes offer the best combination of:
   ✓ Strong competition (high LCC presence)
   ✓ Low market concentration
   ✓ Efficient pricing (good $/mile)
   ✓ Affordable absolute fares


### Score Components Breakdown

This shows how the #1 best route in the market achieves its high affordability score.

In [41]:
# Analyze score components for top routes
top_route = best_routes.iloc[0]['route_directional']
top_route_data = df_scored[df_scored['route_directional'] == top_route].iloc[0]

components = pd.DataFrame({
    'Component': ['Competition\n(LCC Presence)', 'Efficiency\n($/mile)', 'Market Power\n(Low Dominance)', 'Price\n(Absolute Fare)'],
    'Score': [top_route_data['competition_score'], 
              top_route_data['efficiency_score'],
              top_route_data['market_score'],
              top_route_data['price_score']],
    'Max': [25, 25, 25, 25]
})

fig = go.Figure()
fig.add_trace(go.Bar(name='Score', x=components['Component'], y=components['Score'], 
                     text=components['Score'], texttemplate='%{text:.1f}/25',
                     marker_color='lightseagreen'))
fig.add_trace(go.Bar(name='Remaining', x=components['Component'], 
                     y=components['Max'] - components['Score'],
                     marker_color='lightgray'))

fig.update_layout(barmode='stack', 
                  title=f'Score Breakdown: {top_route} (Best Route in Dataset)',
                  yaxis_title='Score (out of 25)',
                  height=500,
                  showlegend=False)
fig.show()

print(f"\n Best Route: {top_route}")
print(f"   Total Score: {top_route_data['affordability_score']:.1f}/100")
print(f"   Average Fare: ${top_route_data['fare']:.2f}")
print(f"   LCC Market Share: {top_route_data['lf_ms']:.1%}")
print(f"   Price per Mile: ${top_route_data['price_per_mile']:.3f}")


 Best Route: Tampa, FL (Metropolitan Area) → Trenton, NJ
   Total Score: 63.9/100
   Average Fare: $98.77
   LCC Market Share: 100.0%
   Price per Mile: $0.103


---

## USER INPUT SECTION - CUSTOMIZE YOUR ANALYSIS HERE!

In [42]:
# ============================================================================
# EDIT THESE VALUES TO ANALYZE YOUR OWN ROUTES
# ============================================================================

# YOUR TRAVEL DETAILS (edit these!)
my_origin = "Los Angeles, CA (Metropolitan Area)"           # [CHANGE] THIS to your departure city
my_destination = "New York City, NY (Metropolitan Area)"        # [CHANGE] THIS to your arrival city

# TRAVEL TIMING (optional - leave as None for recommendations)
departure_quarter = None      # [OPTIONAL] Set to 1, 2, 3, or 4 (or None for recommendation)
return_quarter = None         # [OPTIONAL] Set to 1, 2, 3, or 4 (or None for recommendation)

# ============================================================================
# Validation and display
# ============================================================================

print("YOUR TRAVEL PROFILE:")
print("=" * 70)
print(f"Origin: {my_origin}")
print(f"Destination: {my_destination}")
print("=" * 70)

# Check if cities exist in dataset
origin_exists = my_origin in all_cities
dest_exists = my_destination in all_cities

if not origin_exists:
    print(f"\n WARNING: '{my_origin}' not found in dataset!")
    print("   Please copy a city name from the list above.")
    
if not dest_exists:
    print(f"\n WARNING: '{my_destination}' not found in dataset!")
    print("   Please copy a city name from the list above.")

if origin_exists and dest_exists:
    print("\nBoth cities found! Ready to analyze your route.")
    
print("\n To change your inputs, edit the variables in this cell and re-run it.")

YOUR TRAVEL PROFILE:
Origin: Los Angeles, CA (Metropolitan Area)
Destination: New York City, NY (Metropolitan Area)

Both cities found! Ready to analyze your route.

 To change your inputs, edit the variables in this cell and re-run it.


### Your Route Data

All available data for your selected route:

In [ ]:
# Display all data for the user's route
your_route = f"{my_origin} → {my_destination}"
your_route_data = df_scored[df_scored['route_directional'] == your_route].copy()

if len(your_route_data) > 0:
    print(f"Found {len(your_route_data)} records for {your_route}\n")
    
    # Display the data table
    display(
        your_route_data[[
            'quarter', 'fare', 'nsmiles', 'passengers',
            'fare_per_mile', 'carrier_lg', 'large_ms',
            'carrier_low', 'lf_ms', 'lcc_count',
            'market_structure', 'affordability_score'
        ]]
        .sort_values('quarter')
        .reset_index(drop=True)
        .style
        .format({
            'fare': '${:.2f}',
            'fare_per_mile': '${:.4f}',
            'large_ms': '{:.1%}',
            'lf_ms': '{:.1%}',
            'affordability_score': '{:.1f}',
            'nsmiles': '{:.0f}',
            'passengers': '{:,.0f}'
        })
        .set_caption(f"All data for route: {your_route}")
    )
else:
    print(f"No data found for route: {your_route}")
    print("Please check your origin and destination cities above.")

### Your Route's Score Breakdown

Let's analyze how YOUR specific route performs on each scoring component.

In [44]:
# Analyze score components for YOUR route
your_route = f"{my_origin} → {my_destination}"
your_route_data = df_scored[df_scored['route_directional'] == your_route]

if len(your_route_data) > 0:
    # Get average scores for your route
    your_route_avg = your_route_data.iloc[0]
    
    components = pd.DataFrame({
        'Component': ['Competition\n(LCC Presence)', 'Efficiency\n($/mile)', 'Market Power\n(Low Dominance)', 'Price\n(Absolute Fare)'],
        'Score': [your_route_avg['competition_score'], 
                  your_route_avg['efficiency_score'],
                  your_route_avg['market_score'],
                  your_route_avg['price_score']],
        'Max': [25, 25, 25, 25]
    })
    
    fig = go.Figure()
    fig.add_trace(go.Bar(name='Score', x=components['Component'], y=components['Score'], 
                         text=components['Score'], texttemplate='%{text:.1f}/25',
                         marker_color='mediumpurple'))
    fig.add_trace(go.Bar(name='Remaining', x=components['Component'], 
                         y=components['Max'] - components['Score'],
                         marker_color='lightgray'))
    
    fig.update_layout(barmode='stack', 
                      title=f'Score Breakdown: {your_route} (Your Route)',
                      yaxis_title='Score (out of 25)',
                      height=500,
                      showlegend=False)
    fig.show()
    
    print(f"\n YOUR ROUTE: {your_route}")
    print(f"   Total Score: {your_route_avg['affordability_score']:.1f}/100")
    print(f"   Average Fare: ${your_route_avg['fare']:.2f}")
    print(f"   LCC Market Share: {your_route_avg['lf_ms']:.1%}")
    print(f"   Price per Mile: ${your_route_avg['price_per_mile']:.3f}")
    
    # Compare to dataset's best route
    best_score = df_scored['affordability_score'].max()
    score_diff = your_route_avg['affordability_score'] - best_score
    
    print(f"\n COMPARISON TO BEST ROUTE:")
    if score_diff >= 0:
        print(f"   Your route IS the best route in the dataset!")
    elif score_diff > -10:
        print(f"   Your route is excellent! Only {abs(score_diff):.1f} points below the best.")
    elif score_diff > -20:
        print(f"   Your route is good! {abs(score_diff):.1f} points below the best.")
    else:
        print(f"   Your route scores {abs(score_diff):.1f} points below the best. Check alternatives below!")
else:
    print(f"\n Route '{your_route}' not found in dataset.")
    print("   Please check your origin and destination cities above.")


 YOUR ROUTE: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)
   Total Score: 42.5/100
   Average Fare: $430.38
   LCC Market Share: 22.7%
   Price per Mile: $0.171

 COMPARISON TO BEST ROUTE:
   Your route scores 23.5 points below the best. Check alternatives below!


## Alternative Route Finder

Find cheaper alternatives by considering nearby airports or different city pairs.

In [45]:
def find_alternative_routes(origin, destination, df_data, max_alternatives=10, preferred_quarter=None):
    """
    Find alternative routes and nearby airports that might offer better deals.
    If preferred_quarter is specified, prioritize routes in that quarter.
    """
    # Direct route
    direct_route = f"{origin} → {destination}"
    direct_data = df_data[df_data['route_directional'] == direct_route]
    
    if len(direct_data) == 0:
        # Try reverse
        direct_route = f"{destination} → {origin}"
        direct_data = df_data[df_data['route_directional'] == direct_route]
    
    if len(direct_data) == 0:
        return None, pd.DataFrame()
    
    direct_avg = direct_data.groupby('route_directional').agg({
        'fare': 'mean',
        'affordability_score': 'mean',
        'lf_ms': 'mean',
        'large_ms': 'mean',
        'nsmiles': 'mean'
    }).reset_index()
    
    # Find alternative routes departing from the same origin
    base_filter = (df_data['city1'] == origin) & (df_data['route_directional'] != direct_route)
    
    if preferred_quarter is not None:
        # Try to find alternatives in the same quarter first
        alternative_routes = df_data[base_filter & (df_data['quarter'] == preferred_quarter)].copy()
        
        # If not enough alternatives, expand to adjacent quarters
        if len(alternative_routes) < max_alternatives:
            # Calculate quarter distances (circular: Q1 is close to Q4)
            def quarter_distance(q1, q2):
                diff = abs(q1 - q2)
                return min(diff, 4 - diff)
            
            # Add routes from other quarters, sorted by quarter proximity
            other_quarters = df_data[base_filter & (df_data['quarter'] != preferred_quarter)].copy()
            other_quarters['quarter_distance'] = other_quarters['quarter'].apply(lambda q: quarter_distance(preferred_quarter, q))
            other_quarters = other_quarters.sort_values('quarter_distance')
            alternative_routes = pd.concat([alternative_routes, other_quarters])
    else:
        alternative_routes = df_data[base_filter].copy()
    
    alternatives = alternative_routes.groupby('route_directional').agg({
        'fare': 'mean',
        'affordability_score': 'mean',
        'lf_ms': 'mean',
        'large_ms': 'mean',
        'nsmiles': 'mean',
        'city1': 'first',
        'city2': 'first',
        'quarter': lambda x: ', '.join(sorted(set(f'Q{int(q)}' for q in x)))
    }).reset_index()
    
    # Calculate potential savings
    if len(direct_avg) > 0:
        direct_fare = direct_avg.iloc[0]['fare']
        alternatives['savings'] = direct_fare - alternatives['fare']
        alternatives['savings_pct'] = (alternatives['savings'] / direct_fare) * 100
        
        # Filter to only cheaper alternatives
        alternatives = alternatives[alternatives['savings'] > 0].nlargest(max_alternatives, 'savings')
    
    return direct_avg.iloc[0] if len(direct_avg) > 0 else None, alternatives

# Use the user's input cities
print(f"Analyzing YOUR route: {my_origin} → {my_destination}")
print("="*80)

# Run alternative route analysis for YOUR cities
direct, alternatives = find_alternative_routes(my_origin, my_destination, df_scored, preferred_quarter=departure_quarter)

if direct is not None:
    print(f"\n DIRECT ROUTE: {direct['route_directional']}")
    print(f"   Average Fare: ${direct['fare']:.2f}")
    print(f"   Affordability Score: {direct['affordability_score']:.1f}/100")
    print(f"   Distance: {direct['nsmiles']:.0f} miles")
    print(f"   LCC Presence: {direct['lf_ms']:.1%}")
    
    if len(alternatives) > 0:
        print(f"\n\n FOUND {len(alternatives)} CHEAPER ALTERNATIVES:")
        print("="*80)
        
        for idx, alt in alternatives.head(5).iterrows():
            print(f"\n{idx+1}. {alt['route_directional']}")
            print(f"   Save: ${alt['savings']:.2f} ({alt['savings_pct']:.1f}%)")
            print(f"   Fare: ${alt['fare']:.2f} | Score: {alt['affordability_score']:.1f}/100")
            print(f"   Distance: {alt['nsmiles']:.0f} miles | LCC: {alt['lf_ms']:.1%}")
            
            # Show available quarters
            quarter_info = f"   Quarters: {alt['quarter']}"
            if departure_quarter is not None:
                if f'Q{departure_quarter}' in alt['quarter']:
                    quarter_info += " ✓"
            print(quarter_info)
            
            # Suggest strategy
            other_city = alt['city2']
            print(f"   💡 Strategy: Fly from {my_origin} to {other_city} instead")
    else:
        print("\n This is already one of the best options available!")
else:
    print("Route not found in dataset. Try another city pair.")

Analyzing YOUR route: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)

 DIRECT ROUTE: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)
   Average Fare: $412.23
   Affordability Score: 43.2/100
   Distance: 2510 miles
   LCC Presence: 25.7%


 FOUND 10 CHEAPER ALTERNATIVES:

19. Los Angeles, CA (Metropolitan Area) → Provo, UT
   Save: $318.31 (77.2%)
   Fare: $93.92 | Score: 52.5/100
   Distance: 568 miles | LCC: 31.3%
   Quarters: Q1, Q2, Q3, Q4
   💡 Strategy: Fly from Los Angeles, CA (Metropolitan Area) to Provo, UT instead

26. Los Angeles, CA (Metropolitan Area) → San Francisco, CA (Metropolitan Area)
   Save: $259.71 (63.0%)
   Fare: $152.52 | Score: 42.6/100
   Distance: 372 miles | LCC: 36.8%
   Quarters: Q1, Q2, Q3, Q4
   💡 Strategy: Fly from Los Angeles, CA (Metropolitan Area) to San Francisco, CA (Metropolitan Area) instead

21. Los Angeles, CA (Metropolitan Area) → Reno, NV
   Save: $259.61 (63.0%)
   Fare: $152.62 | Scor

## Optimal Booking Window

Identify the best quarters to travel based on historical patterns.

In [46]:
# Filter data for user's specific route
user_route = df_scored[
    ((df_scored['city1'] == my_origin) & (df_scored['city2'] == my_destination))
].copy()

if len(user_route) == 0:
    print(f"No data found for route {my_origin} → {my_destination}")
    print("Please check your origin/destination cities.")
else:
    # Analyze quarterly patterns for this specific route
    route_quarterly = user_route.groupby('quarter').agg({
        'fare': ['mean', 'min', 'max'],
        'affordability_score': 'mean',
        'passengers': 'sum'
    }).reset_index()
    
    route_quarterly.columns = ['Quarter', 'Mean Fare', 'Min Fare', 'Max Fare', 'Avg Score', 'Total Passengers']
    
    quarter_labels = {1: 'Q1\n(Jan-Mar)', 2: 'Q2\n(Apr-Jun)', 3: 'Q3\n(Jul-Sep)', 4: 'Q4\n(Oct-Dec)'}
    route_quarterly['Quarter Label'] = route_quarterly['Quarter'].map(quarter_labels)
    
    # ========================================================================
    # CONDITIONAL LOGIC: Recommendations vs. Cost Estimates
    # ========================================================================
    
    if departure_quarter is None and return_quarter is None:
        # USER DID NOT SPECIFY QUARTERS - PROVIDE RECOMMENDATIONS
        print("\n" + "="*70)
        print(" PERSONALIZED QUARTER RECOMMENDATIONS")
        print(f"    Route: {my_origin} → {my_destination}")
        print("="*70)
        
        best_q = route_quarterly.loc[route_quarterly['Mean Fare'].idxmin()]
        worst_q = route_quarterly.loc[route_quarterly['Mean Fare'].idxmax()]
        savings_potential = worst_q['Mean Fare'] - best_q['Mean Fare']
        
        print(f"\n✅ BEST QUARTER TO TRAVEL: {quarter_labels[best_q['Quarter']].replace(chr(10), ' ')}")
        print(f"   Expected Fare: ${best_q['Mean Fare']:.2f}")
        print(f"   Fare Range: ${best_q['Min Fare']:.2f} - ${best_q['Max Fare']:.2f}")
        print(f"   Affordability Score: {best_q['Avg Score']:.1f}/100")
        
        print(f"\n❌ AVOID: {quarter_labels[worst_q['Quarter']].replace(chr(10), ' ')}")
        print(f"   Expected Fare: ${worst_q['Mean Fare']:.2f}")
        print(f"   💰 Potential Savings: ${savings_potential:.2f} ({savings_potential/worst_q['Mean Fare']*100:.1f}% cheaper)")
        
        print("\n📊 ALL QUARTERS FOR YOUR ROUTE:")
        for _, row in route_quarterly.sort_values('Mean Fare').iterrows():
            marker = "✅" if row['Quarter'] == best_q['Quarter'] else "❌" if row['Quarter'] == worst_q['Quarter'] else "  "
            print(f"   {marker} {quarter_labels[row['Quarter']].replace(chr(10), ' '):20} ${row['Mean Fare']:7.2f}  (Score: {row['Avg Score']:.1f}/100)")
    
    else:
        # USER SPECIFIED QUARTERS - SHOW COST ESTIMATES
        print("\n" + "="*70)
        print(" 💰 YOUR TRAVEL COST ESTIMATE")
        print(f"    Route: {my_origin} → {my_destination}")
        print("="*70)
        
        total_cost = 0
        
        if departure_quarter is not None:
            dep_data = route_quarterly[route_quarterly['Quarter'] == departure_quarter]
            if len(dep_data) > 0:
                dep_fare = dep_data.iloc[0]['Mean Fare']
                dep_score = dep_data.iloc[0]['Avg Score']
                dep_range_low = dep_data.iloc[0]['Min Fare']
                dep_range_high = dep_data.iloc[0]['Max Fare']
                total_cost += dep_fare
                
                print(f"\n✈️  DEPARTURE: {quarter_labels[departure_quarter].replace(chr(10), ' ')}")
                print(f"   Expected Fare: ${dep_fare:.2f}")
                print(f"   Typical Range: ${dep_range_low:.2f} - ${dep_range_high:.2f}")
                print(f"   Affordability: {dep_score:.1f}/100")
            else:
                print(f"\n⚠️  No data for departure in Q{departure_quarter}")
        
        if return_quarter is not None:
            ret_data = route_quarterly[route_quarterly['Quarter'] == return_quarter]
            if len(ret_data) > 0:
                ret_fare = ret_data.iloc[0]['Mean Fare']
                ret_score = ret_data.iloc[0]['Avg Score']
                ret_range_low = ret_data.iloc[0]['Min Fare']
                ret_range_high = ret_data.iloc[0]['Max Fare']
                total_cost += ret_fare
                
                print(f"\n🏠 RETURN: {quarter_labels[return_quarter].replace(chr(10), ' ')}")
                print(f"   Expected Fare: ${ret_fare:.2f}")
                print(f"   Typical Range: ${ret_range_low:.2f} - ${ret_range_high:.2f}")
                print(f"   Affordability: {ret_score:.1f}/100")
            else:
                print(f"\n⚠️  No data for return in Q{return_quarter}")
        
        if total_cost > 0:
            print(f"\n💵 TOTAL ESTIMATED COST: ${total_cost:.2f}")
            
            # Compare to best option
            best_q = route_quarterly.loc[route_quarterly['Mean Fare'].idxmin()]
            if departure_quarter == best_q['Quarter'] or return_quarter == best_q['Quarter']:
                print(f"   ✅ You're traveling in the cheapest quarter!")
            else:
                potential_savings = total_cost - (best_q['Mean Fare'] * 2 if return_quarter is not None and departure_quarter is not None else best_q['Mean Fare'])
                if potential_savings > 0:
                    print(f"   💡 TIP: Traveling in {quarter_labels[best_q['Quarter']].replace(chr(10), ' ')} could save ~${potential_savings:.2f}")
    
    # Visualization
    fig = make_subplots(rows=1, cols=2, 
                        subplot_titles=(f'Average Fare by Quarter', 
                                       'Affordability Score by Quarter'),
                        specs=[[{"type": "bar"}, {"type": "bar"}]],
                        horizontal_spacing=0.12)
    
    # Highlight user's selected quarters
    if departure_quarter is not None or return_quarter is not None:
        colors = []
        for q in route_quarterly['Quarter']:
            if q in [departure_quarter, return_quarter]:
                colors.append('#9b59b6')  # Purple for selected
            elif q == route_quarterly.loc[route_quarterly['Mean Fare'].idxmin(), 'Quarter']:
                colors.append('#2ecc71')  # Green for cheapest
            else:
                colors.append('#3498db')  # Blue for others
    else:
        colors = ['#2ecc71' if x == route_quarterly['Mean Fare'].min() else 
                 '#e74c3c' if x == route_quarterly['Mean Fare'].max() else 
                 '#3498db' for x in route_quarterly['Mean Fare']]
    
    fig.add_trace(
        go.Bar(x=route_quarterly['Quarter Label'], y=route_quarterly['Mean Fare'],
               text=route_quarterly['Mean Fare'], texttemplate='$%{text:.2f}',
               marker_color=colors,
               name='Mean Fare'),
        row=1, col=1
    )
    
    score_colors = ['#2ecc71' if x == route_quarterly['Avg Score'].max() else 
                   '#e74c3c' if x == route_quarterly['Avg Score'].min() else 
                   '#3498db' for x in route_quarterly['Avg Score']]
    
    fig.add_trace(
        go.Bar(x=route_quarterly['Quarter Label'], y=route_quarterly['Avg Score'],
               text=route_quarterly['Avg Score'], texttemplate='%{text:.1f}',
               marker_color=score_colors,
               name='Avg Score'),
        row=1, col=2
    )
    
    fig.update_xaxes(title_text="Quarter", row=1, col=1)
    fig.update_xaxes(title_text="Quarter", row=1, col=2)
    fig.update_yaxes(title_text="Average Fare ($)", row=1, col=1)
    fig.update_yaxes(title_text="Affordability Score", row=1, col=2)
    
    fig.update_layout(
        height=500, 
        showlegend=False, 
        title_text=f"Optimal Travel Timing Analysis: {my_origin} ↔ {my_destination}",
        title_x=0.5,
        title_xanchor='center',
        margin=dict(t=80, b=50)
    )
    fig.show()


 PERSONALIZED QUARTER RECOMMENDATIONS
    Route: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)

✅ BEST QUARTER TO TRAVEL: Q1 (Jan-Mar)
   Expected Fare: $378.21
   Fare Range: $315.77 - $415.08
   Affordability Score: 43.8/100

❌ AVOID: Q4 (Oct-Dec)
   Expected Fare: $441.21
   💰 Potential Savings: $63.00 (14.3% cheaper)

📊 ALL QUARTERS FOR YOUR ROUTE:
   ✅ Q1 (Jan-Mar)         $ 378.21  (Score: 43.8/100)
      Q3 (Jul-Sep)         $ 401.46  (Score: 43.4/100)
      Q2 (Apr-Jun)         $ 432.60  (Score: 42.9/100)
   ❌ Q4 (Oct-Dec)         $ 441.21  (Score: 42.4/100)


# ═══════════════════════════════════════════════════════════════════════
# PANEL 1 — Route Explorer
# Select an origin city to browse all routes, fares, and market info.
# ═══════════════════════════════════════════════════════════════════════

In [47]:


city_dropdown = widgets.Dropdown(
    options=all_cities,
    description='Origin city:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

sort_dropdown = widgets.Dropdown(
    options=[
        ('Fare (low → high)', 'fare_asc'),
        ('Fare (high → low)', 'fare_desc'),
        ('Distance (short → long)', 'dist_asc'),
        ('Distance (long → short)', 'dist_desc'),
        ('Passengers (most first)', 'pax_desc'),
    ],
    value='fare_asc',
    description='Sort by:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='40%')
)

route_output = widgets.Output()

def update_route_explorer(*_):
    city = city_dropdown.value
    sort_key = sort_dropdown.value

    subset = df[(df['city1'] == city) | (df['city2'] == city)].copy()

    sort_map = {
        'fare_asc': ('fare', True),
        'fare_desc': ('fare', False),
        'dist_asc': ('nsmiles', True),
        'dist_desc': ('nsmiles', False),
        'pax_desc': ('passengers', False),
    }
    col, asc = sort_map[sort_key]
    subset = subset.sort_values(col, ascending=asc)

    with route_output:
        clear_output(wait=True)
        if subset.empty:
            print(f"No routes found for {city}")
            return

        print(f"Routes from/to: {city}  ({len(subset)} records)\n")
        display(
            subset[['city1', 'city2', 'nsmiles', 'passengers', 'fare',
                     'fare_per_mile', 'carrier_lg', 'large_ms',
                     'carrier_low', 'lf_ms', 'market_structure',
                     'lcc_present', 'quarter']]
            .reset_index(drop=True)
            .style
            .format({
                'fare': '${:.2f}',
                'fare_per_mile': '${:.4f}',
                'large_ms': '{:.1%}',
                'lf_ms': '{:.1%}',
            })
            .set_caption(f"Route details for {city}")
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        top = subset.drop_duplicates('route').nlargest(10, 'fare')
        sns.barplot(data=top, y='route', x='fare', hue='route',
                    palette='YlOrRd_r', ax=axes[0], legend=False)
        axes[0].set_title('Top 10 Most Expensive Routes', fontweight='bold')
        axes[0].set_xlabel('Fare ($)')
        axes[0].set_ylabel('')

        all_structures = ['Competitive (<50%)', 'Moderate (50-75%)', 'Dominant (>75%)']
        structure_counts = subset['market_structure'].value_counts()
        structure_counts = structure_counts.reindex(all_structures, fill_value=0)
        colors = sns.color_palette('Set2', len(all_structures))
        bars = axes[1].barh(all_structures, structure_counts.values, color=colors)
        axes[1].set_title('Market Structure Breakdown', fontweight='bold')
        axes[1].set_xlabel('Number of Routes')
        total = structure_counts.sum()
        for bar, count in zip(bars, structure_counts.values):
            pct = count / total * 100 if total > 0 else 0
            label = f'{count}  ({pct:.0f}%)'
            axes[1].text(bar.get_width() + max(total * 0.02, 0.3),
                         bar.get_y() + bar.get_height() / 2,
                         label, va='center', fontsize=10)

        plt.tight_layout()
        plt.show()

city_dropdown.observe(update_route_explorer, names='value')
sort_dropdown.observe(update_route_explorer, names='value')

display(widgets.HBox([city_dropdown, sort_dropdown]))
display(route_output)
update_route_explorer()

Output()

# ═══════════════════════════════════════════════════════════════════════
# PANEL 2 — LCC Penetration vs Fare
# Interactive scatter showing how LCC market share relates to fares.
# Filter by distance range and quarter.
# ═══════════════════════════════════════════════════════════════════════

In [48]:


dist_slider = widgets.IntRangeSlider(
    value=[int(df['nsmiles'].min()), int(df['nsmiles'].max())],
    min=int(df['nsmiles'].min()),
    max=int(df['nsmiles'].max()),
    step=50,
    description='Distance (mi):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

quarter_selector = widgets.SelectMultiple(
    options=[1, 2, 3, 4],
    value=[1, 2, 3, 4],
    description='Quarter(s):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='20%', height='90px')
)

metric_toggle = widgets.ToggleButtons(
    options=['fare', 'fare_per_mile'],
    value='fare',
    description='Y-axis:',
    style={'description_width': 'initial'},
    button_style='info'
)

lcc_output = widgets.Output()

def update_lcc_scatter(*_):
    lo, hi = dist_slider.value
    quarters = list(quarter_selector.value)
    y_col = metric_toggle.value

    subset = df[
        (df['nsmiles'] >= lo) &
        (df['nsmiles'] <= hi) &
        (df['quarter'].isin(quarters))
    ].dropna(subset=['lf_ms', y_col])

    with lcc_output:
        clear_output(wait=True)

        if subset.empty:
            print("No data matches the current filters.")
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].scatter(subset['lf_ms'], subset[y_col],
                        alpha=0.25, s=12, c='steelblue')
        z = np.polyfit(subset['lf_ms'], subset[y_col], 1)
        p = np.poly1d(z)
        x_line = np.linspace(subset['lf_ms'].min(), subset['lf_ms'].max(), 100)
        axes[0].plot(x_line, p(x_line), color='red', linewidth=2,
                     linestyle='--', label='Trend')
        y_label = 'Fare ($)' if y_col == 'fare' else 'Fare per Mile ($)'
        axes[0].set_xlabel('LCC Market Share', fontsize=12)
        axes[0].set_ylabel(y_label, fontsize=12)
        axes[0].set_title('LCC Penetration vs Fare', fontsize=14, fontweight='bold')
        axes[0].legend()

        corr = subset['lf_ms'].corr(subset[y_col])
        axes[0].text(0.02, 0.95, f'r = {corr:.3f}',
                     transform=axes[0].transAxes, fontsize=11,
                     verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        bins = pd.cut(subset['lf_ms'], bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0],
                       labels=['0-10%', '10-20%', '20-30%', '30-50%', '50-100%'])
        bin_means = subset.groupby(bins, observed=True)[y_col].mean()
        bin_means.plot(kind='bar', ax=axes[1], color=sns.color_palette('viridis', len(bin_means)))
        axes[1].set_title(f'Avg {y_label} by LCC Share Bucket', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('LCC Market Share Bucket', fontsize=12)
        axes[1].set_ylabel(y_label, fontsize=12)
        axes[1].tick_params(axis='x', rotation=0)

        for i, v in enumerate(bin_means):
            fmt = f'${v:.2f}' if y_col == 'fare' else f'${v:.4f}'
            axes[1].text(i, v + v * 0.01, fmt, ha='center', fontsize=9)

        plt.tight_layout()
        plt.show()

        print(f"\nShowing {len(subset):,} routes  |  "
              f"Distance: {lo}–{hi} mi  |  "
              f"Quarter(s): {quarters}")

dist_slider.observe(update_lcc_scatter, names='value')
quarter_selector.observe(update_lcc_scatter, names='value')
metric_toggle.observe(update_lcc_scatter, names='value')

display(widgets.VBox([
    metric_toggle,
    widgets.HBox([dist_slider, quarter_selector]),
]))
display(lcc_output)
update_lcc_scatter()

Output()

# ═══════════════════════════════════════════════════════════════════════
# PANEL 3 — Market Structure Comparison
# Compare fares across market structures with interactive filters.
# ═══════════════════════════════════════════════════════════════════════


In [49]:

ms_metric = widgets.ToggleButtons(
    options=[('Average Fare', 'fare'), ('Fare per Mile', 'fare_per_mile')],
    value='fare',
    description='Metric:',
    style={'description_width': 'initial'},
    button_style='info'
)

ms_lcc_filter = widgets.ToggleButtons(
    options=[('All Routes', 'all'), ('LCC Present', 'lcc'), ('Legacy Only', 'legacy')],
    value='all',
    description='Filter:',
    style={'description_width': 'initial'},
    button_style='warning'
)

ms_quarter = widgets.Dropdown(
    options=[('All Quarters', 0), ('Q1', 1), ('Q2', 2), ('Q3', 3), ('Q4', 4)],
    value=0,
    description='Quarter:',
    style={'description_width': 'initial'},
)

ms_output = widgets.Output()

def update_market_structure(*_):
    metric = ms_metric.value
    lcc_filter = ms_lcc_filter.value
    q = ms_quarter.value

    subset = df.copy()

    if lcc_filter == 'lcc':
        subset = subset[subset['lcc_present']]
    elif lcc_filter == 'legacy':
        subset = subset[~subset['lcc_present']]

    if q > 0:
        subset = subset[subset['quarter'] == q]

    with ms_output:
        clear_output(wait=True)

        if subset.empty:
            print("No data matches the current filters.")
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: by market structure
        struct_means = subset.groupby('market_structure', observed=True)[metric].mean().reset_index()
        sns.barplot(data=struct_means, x='market_structure', y=metric,
                    hue='market_structure', palette='Reds', ax=axes[0], legend=False)
        y_label = 'Average Fare ($)' if metric == 'fare' else 'Fare per Mile ($)'
        axes[0].set_title(f'{y_label} by Market Structure', fontsize=14, fontweight='bold')
        axes[0].set_ylabel(y_label, fontsize=12)
        axes[0].set_xlabel('')
        for c in axes[0].containers:
            fmt = '${:.2f}' if metric == 'fare' else '${:.4f}'
            axes[0].bar_label(c, fmt=fmt, padding=3)

        # Right: LCC vs Legacy on dominant routes
        dominant = subset[subset['market_structure'] == 'Dominant (>75%)']
        if not dominant.empty:
            dom_means = dominant.groupby('dominant_type')[metric].mean().reset_index()
            sns.barplot(data=dom_means, x='dominant_type', y=metric,
                        hue='dominant_type', palette='coolwarm', ax=axes[1], legend=False)
            axes[1].set_title(f'{y_label}: LCC vs Legacy (Dominant Routes)',
                              fontsize=14, fontweight='bold')
            axes[1].set_ylabel(y_label, fontsize=12)
            axes[1].set_xlabel('')
            for c in axes[1].containers:
                fmt = '${:.2f}' if metric == 'fare' else '${:.4f}'
                axes[1].bar_label(c, fmt=fmt, padding=3)
        else:
            axes[1].text(0.5, 0.5, 'No dominant routes\nfor this filter',
                         ha='center', va='center', fontsize=14, transform=axes[1].transAxes)
            axes[1].set_title('Dominant Route Breakdown', fontweight='bold')

        plt.tight_layout()
        plt.show()

        print(f"\nShowing {len(subset):,} routes  |  "
              f"Filter: {lcc_filter}  |  "
              f"Quarter: {'All' if q == 0 else f'Q{q}'}")

        summary = subset.groupby('market_structure', observed=True)[metric].describe()[
            ['count', 'mean', 'std', 'min', 'max']
        ]
        display(summary.style.format({
            'mean': '{:.4f}', 'std': '{:.4f}', 'min': '{:.4f}', 'max': '{:.4f}'
        }))

ms_metric.observe(update_market_structure, names='value')
ms_lcc_filter.observe(update_market_structure, names='value')
ms_quarter.observe(update_market_structure, names='value')

display(widgets.VBox([ms_metric, widgets.HBox([ms_lcc_filter, ms_quarter])]))
display(ms_output)
update_market_structure()

Output()

# ═══════════════════════════════════════════════════════════════════════
# PANEL 4 — Fare Predictor
# Use the trained Random Forest model to predict fares for custom
# route parameters.  Adjust the sliders and see the result update.
# ═══════════════════════════════════════════════════════════════════════

In [50]:


pred_distance = widgets.IntSlider(
    value=1000, min=50, max=5000, step=50,
    description='Distance (mi):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

pred_large_ms = widgets.FloatSlider(
    value=0.50, min=0.0, max=1.0, step=0.01,
    description='Largest carrier share:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%'),
    readout_format='.0%'
)

pred_lf_ms = widgets.FloatSlider(
    value=0.20, min=0.0, max=1.0, step=0.01,
    description='LCC market share:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%'),
    readout_format='.0%'
)

pred_passengers = widgets.IntSlider(
    value=3000, min=100, max=20000, step=100,
    description='Passengers:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

pred_quarter = widgets.Dropdown(
    options=[('Q1', 1), ('Q2', 2), ('Q3', 3), ('Q4', 4)],
    value=2,
    description='Quarter:',
    style={'description_width': 'initial'},
)

pred_output = widgets.Output()

def update_prediction(*_):
    input_df = pd.DataFrame([{
        'nsmiles': pred_distance.value,
        'large_ms': pred_large_ms.value,
        'lf_ms': pred_lf_ms.value,
        'passengers': pred_passengers.value,
        'quarter': pred_quarter.value,
    }])

    predicted_fare = rf_model.predict(input_df)[0]

    similar = df[
        (df['nsmiles'].between(pred_distance.value * 0.8, pred_distance.value * 1.2)) &
        (df['quarter'] == pred_quarter.value)
    ]
    actual_median = similar['fare'].median() if not similar.empty else None

    with pred_output:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8, 4))

        bars = ax.barh(
            ['Model Prediction', 'Actual Median\n(similar routes)'],
            [predicted_fare, actual_median if actual_median else 0],
            color=['steelblue', 'coral'],
            height=0.4
        )

        ax.set_xlabel('Fare ($)', fontsize=12)
        ax.set_title('Predicted vs Actual Fares', fontsize=14, fontweight='bold')
        ax.set_xlim(0, max(predicted_fare, actual_median or 0) * 1.3)

        ax.bar_label(bars, fmt='$%.2f', padding=5, fontsize=11)

        if actual_median is None:
            ax.text(0.5, 0.15,
                    '(No similar routes in dataset for comparison)',
                    ha='center', transform=ax.transAxes,
                    fontsize=10, color='gray')

        plt.tight_layout()
        plt.show()

        fare_per_mile = predicted_fare / pred_distance.value if pred_distance.value > 0 else 0
        ms_label = (
            'Competitive' if pred_large_ms.value < 0.5
            else 'Moderate' if pred_large_ms.value < 0.75
            else 'Dominant'
        )
        lcc_label = 'High' if pred_lf_ms.value > 0.3 else 'Moderate' if pred_lf_ms.value > 0.1 else 'Low'

        print(f"  Predicted fare:       ${predicted_fare:.2f}")
        print(f"  Predicted $/mile:     ${fare_per_mile:.4f}")
        print(f"  Market structure:     {ms_label}")
        print(f"  LCC penetration:      {lcc_label}")
        if actual_median is not None:
            diff = predicted_fare - actual_median
            print(f"  vs. actual median:    {'+' if diff > 0 else ''}{diff:.2f} "
                  f"({'above' if diff > 0 else 'below'} median)")
            print(f"  Similar routes found: {len(similar):,}")

for w in [pred_distance, pred_large_ms, pred_lf_ms, pred_passengers, pred_quarter]:
    w.observe(update_prediction, names='value')

display(widgets.VBox([
    pred_distance, pred_large_ms, pred_lf_ms,
    pred_passengers, pred_quarter
]))
display(pred_output)
update_prediction()

Output()

# ═══════════════════════════════════════════════════════════════════════
# PANEL 5 — LCC Effect on Fare at a Fixed Distance
# Pick a distance. See how the average fare drops as the number of
# LCC carriers on the route increases from 0 → 1 → 2.
# ═══════════════════════════════════════════════════════════════════════

In [51]:


lcc_distance = widgets.IntSlider(
    value=600, min=100, max=1500, step=50,
    description='Route distance (mi):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%')
)

lcc_window_label = widgets.HTML(
    value='<i style="color:#888; font-size:12px;">'
          'Short-to-medium haul routes (≤1,500 mi). '
          'Routes within ±100 miles of the selected distance are included.</i>'
)

lcc_effect_output = widgets.Output()

WINDOW = 100  # ±miles around selected distance

def update_lcc_effect(*_):
    dist = lcc_distance.value
    lo, hi = dist - WINDOW, dist + WINDOW

    subset = df[
        (df['nsmiles'] >= lo) & (df['nsmiles'] <= hi)
    ].dropna(subset=['fare'])

    with lcc_effect_output:
        clear_output(wait=True)

        if subset.empty:
            print(f"No routes found near {dist} miles.")
            return

        # Compute averages for each LCC count (always show 0, 1, 2)
        all_counts = [0, 1, 2]
        fare_means = []
        fpm_means = []
        sample_sizes = []

        for c in all_counts:
            group = subset[subset['lcc_count'] == c]
            fare_means.append(group['fare'].mean() if not group.empty else None)
            fpm_means.append(group['fare_per_mile'].mean() if not group.empty else None)
            sample_sizes.append(len(group))

        colors = ['#ef5350', '#ff9800', '#4caf50']
        x_labels = ['0 LCCs\n(Legacy only)', '1 LCC', '2 LCCs']

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # ── Left: Average Fare stepping down ──
        valid_fares = [(c, f, n) for c, f, n in zip(all_counts, fare_means, sample_sizes) if f is not None]
        if valid_fares:
            counts_v, fares_v, sizes_v = zip(*valid_fares)
            bar_colors = [colors[c] for c in counts_v]
            bar_labels = [x_labels[c] for c in counts_v]

            bars = axes[0].bar(bar_labels, fares_v, color=bar_colors,
                               width=0.45, edgecolor='white', linewidth=2)
            for bar, val, n in zip(bars, fares_v, sizes_v):
                axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                             f'${val:.2f}\n({n:,} routes)',
                             ha='center', va='bottom', fontsize=11, fontweight='bold')

            if len(valid_fares) >= 2 and 0 in counts_v:
                first_fare = fares_v[0]
                last_fare = fares_v[-1]
                diff = last_fare - first_fare
                pct = abs(diff) / first_fare * 100
                if diff < 0:
                    label = f'  −${abs(diff):.0f} ({pct:.1f}% cheaper)'
                    color = '#2e7d32'
                else:
                    label = f'  +${diff:.0f} ({pct:.1f}% more expensive)'
                    color = '#c62828'
                axes[0].annotate(
                    label,
                    xy=(len(valid_fares) - 1, last_fare),
                    xytext=(len(valid_fares) - 0.5, (first_fare + last_fare) / 2),
                    fontsize=11, color=color, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=color, lw=2),
                    va='center'
                )

        axes[0].set_title(f'Average Fare at ~{dist} Miles', fontsize=14, fontweight='bold')
        axes[0].set_ylabel('Average Fare ($)', fontsize=12)
        axes[0].set_xlabel('')

        # ── Right: Fare per Mile stepping down ──
        valid_fpm = [(c, f, n) for c, f, n in zip(all_counts, fpm_means, sample_sizes) if f is not None]
        if valid_fpm:
            counts_v, fpms_v, sizes_v = zip(*valid_fpm)
            bar_colors = [colors[c] for c in counts_v]
            bar_labels = [x_labels[c] for c in counts_v]

            bars = axes[1].bar(bar_labels, fpms_v, color=bar_colors,
                               width=0.45, edgecolor='white', linewidth=2)
            for bar, val, n in zip(bars, fpms_v, sizes_v):
                axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                             f'${val:.4f}\n({n:,} routes)',
                             ha='center', va='bottom', fontsize=11, fontweight='bold')

            if len(valid_fpm) >= 2 and 0 in counts_v:
                first_fpm = fpms_v[0]
                last_fpm = fpms_v[-1]
                diff = last_fpm - first_fpm
                pct = abs(diff) / first_fpm * 100
                if diff < 0:
                    label = f'  −${abs(diff):.4f} ({pct:.1f}% cheaper)'
                    color = '#2e7d32'
                else:
                    label = f'  +${diff:.4f} ({pct:.1f}% more expensive)'
                    color = '#c62828'
                axes[1].annotate(
                    label,
                    xy=(len(valid_fpm) - 1, last_fpm),
                    xytext=(len(valid_fpm) - 0.5, (first_fpm + last_fpm) / 2),
                    fontsize=11, color=color, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=color, lw=2),
                    va='center'
                )

        axes[1].set_title(f'Fare per Mile at ~{dist} Miles', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('Fare per Mile ($)', fontsize=12)
        axes[1].set_xlabel('')

        plt.tight_layout()
        plt.show()

        # Summary
        avg_dist = subset['nsmiles'].mean()
        print(f"  Distance selected:    {dist} mi  (showing routes {lo}–{hi} mi)")
        print(f"  Avg distance in data: {avg_dist:.0f} mi")
        print(f"  Total routes:         {len(subset):,}")
        for c, f, fpm, n in zip(all_counts, fare_means, fpm_means, sample_sizes):
            if f is not None:
                print(f"  {c} LCC(s):  ${f:.2f} avg fare  |  ${fpm:.4f}/mi  |  {n:,} routes")
            else:
                print(f"  {c} LCC(s):  no data")

lcc_distance.observe(update_lcc_effect, names='value')

display(widgets.VBox([lcc_distance, lcc_window_label]))
display(lcc_effect_output)
update_lcc_effect()

Output()

# ═══════════════════════════════════════════════════════════════════════
# KEY FINDINGS — Executive Summary
# Headline statistics from the market structure & pricing analysis.
# ═══════════════════════════════════════════════════════════════════════

In [ ]:


from IPython.display import HTML

# 1. LCC fare-per-mile gap
fpm_lcc = df[df['lcc_present']]['fare_per_mile'].mean()
fpm_legacy = df[~df['lcc_present']]['fare_per_mile'].mean()
lcc_gap_pct = (fpm_legacy - fpm_lcc) / fpm_legacy * 100

# 2. Dominant route gap: LCC-dominated vs Legacy-dominated
dom = df[df['market_structure'] == 'Dominant (>75%)']
fpm_dom_lcc = dom[dom['dominant_type'] == 'LCC']['fare_per_mile'].mean()
fpm_dom_legacy = dom[dom['dominant_type'] == 'Legacy']['fare_per_mile'].mean()

# 3. Feature importance from the trained model
importances = dict(zip(features, rf_model.feature_importances_))
top_feature = max(importances, key=importances.get)
top_importance = importances[top_feature] * 100

# 4. LCC count effect
fare_by_lcc_count = df.groupby('lcc_count')['fare'].mean()
fare_0_lcc = fare_by_lcc_count.get(0, 0)
fare_2_lcc = fare_by_lcc_count.get(2, 0)
lcc_count_saving = fare_0_lcc - fare_2_lcc

card_style = (
    "display:inline-block; width:23%; min-width:180px; margin:0.5%; "
    "padding:16px; border-radius:10px; text-align:center; "
    "vertical-align:top; font-family:sans-serif;"
)

html = f"""
<div style="margin-bottom:10px">
<h2 style="font-family:sans-serif; margin-bottom:4px;">Key Findings: Market Structure & Pricing</h2>
<p style="font-family:sans-serif; color:#666; margin-top:0;">
Based on {len(df):,} route-quarter observations
</p>
</div>

<div style="display:flex; flex-wrap:wrap; gap:8px; justify-content:center;">

<div style="{card_style} background:#e8f5e9; border:2px solid #4caf50;">
<div style="font-size:28px; font-weight:bold; color:#2e7d32;">{lcc_gap_pct:.1f}%</div>
<div style="font-size:13px; color:#333; margin-top:4px;">
Lower fare-per-mile on<br>routes with LCC presence
</div>
<div style="font-size:11px; color:#888; margin-top:6px;">
${fpm_lcc:.4f} vs ${fpm_legacy:.4f} /mi
</div>
</div>

<div style="{card_style} background:#fff3e0; border:2px solid #ff9800;">
<div style="font-size:28px; font-weight:bold; color:#e65100;">${fpm_dom_legacy:.2f}</div>
<div style="font-size:13px; color:#333; margin-top:4px;">
Fare/mi on legacy-dominated<br>routes vs ${fpm_dom_lcc:.2f} for LCC
</div>
<div style="font-size:11px; color:#888; margin-top:6px;">
Among dominant (&gt;75% share) routes
</div>
</div>

<div style="{card_style} background:#e3f2fd; border:2px solid #1976d2;">
<div style="font-size:28px; font-weight:bold; color:#1565c0;">{top_importance:.1f}%</div>
<div style="font-size:13px; color:#333; margin-top:4px;">
Fare variance explained by<br>distance (top predictor)
</div>
<div style="font-size:11px; color:#888; margin-top:6px;">
Random Forest R² = {rf_model.score(X_test, y_test):.3f}
</div>
</div>

<div style="{card_style} background:#fce4ec; border:2px solid #e91e63;">
<div style="font-size:28px; font-weight:bold; color:#c62828;">${lcc_count_saving:.0f}</div>
<div style="font-size:13px; color:#333; margin-top:4px;">
Avg fare saved when 2 LCCs<br>present vs none
</div>
<div style="font-size:11px; color:#888; margin-top:6px;">
${fare_0_lcc:.0f} (0 LCCs) → ${fare_2_lcc:.0f} (2 LCCs)
</div>
</div>

</div>
"""

display(HTML(html))